# m_viz: MATPOWER case geovisualization
* conda: h-py312-basic

Parses MATPOWER `.m` case files (ACOPF or PF solutions) into normalized pandas DataFrames
(bus, gen, branch) and produces interactive Plotly geo maps. Geographic coordinates come from a
`.gic` or `.AUX` file. Generator fuel labels and multi-circuit branch data are read from the `.m`
file.

The map visualizes the solution stored in the `.m` file: bus loads, dispatched generation (marker
size proportional to MW), and branch loading percentage (viridis color scale).

Optionally, point `GRIDKIT_REPO` to a local GridKit repository clone to augment hover labels with
GridKit-assigned bus and branch IDs.

Key steps:
- **Colocated bus splitting**: buses sharing identical lat/lon are spread on a small circle so each is individually hoverable.
- **Generator fan-out**: when a bus has more than one generator, all generators at that bus are spread radially so each marker is individually hoverable; thin connector lines link them back to the bus. Optionally, single-generator buses can also be offset (`FAN_SINGLE_GEN = True`) to keep the bus marker visible underneath.
- **Fault bus overlay**: optionally highlights a faulted bus with a red marker.
- **Branch loading coloring**: branches colored by loading percentage when flow data is available.

Currently supported cases: **Hawaii40**, **Illinois (ACTIVSg200)**, **Texas (ACTIVSg2000)**, **WECC (ACTIVSg10k)**.


In [ ]:
from pathlib import Path
import importlib
import os
import sys

# py-utils is one level up from the notebooks/ directory
_py_utils = Path.cwd().parent / "py-utils"
if str(_py_utils) not in sys.path:
    sys.path.insert(0, str(_py_utils))

import pandas as pd

import m_viz_utils
from m_viz_utils import read_matpower_case, summarize_case, validation_report_df

## environment and display setup

Keep the notebook lightweight for now. We only need enough setup to inspect the `.m` case tables cleanly.

In [ ]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

onkestrel = "NREL_CLUSTER" in os.environ and os.environ["NREL_CLUSTER"] == "kestrel"
print(f"onkestrel: {onkestrel}")

# configure input paths: select case

Download TAMU synthetic grid cases from
[Texas A&M Electric Grid Test Cases](https://electricgrids.engr.tamu.edu/electric-grid-test-cases/).
After unzipping, point `CASE_DATA_DIR` to the folder containing the extracted files.

Files used (Hawaii40 example):

| File | Used for |
|---|---|
| `Hawaii40_20231026.m` | MATPOWER case (buses, generators, branches) |
| `Hawaii40_GIC_data.gic` | Geographic coordinates (preferred) |
| `Hawaii40_20231026.AUX` | Geographic coordinates (fallback if no `.gic`) |

Files used (ACTIVSg200 / Illinois example):

| File | Used for |
|---|---|
| `case_ACTIVSg200.m` | MATPOWER case (buses, generators, branches) |
| `ACTIVSg200_GIC_data.gic` | Geographic coordinates (preferred) |
| `ACTIVSg200.AUX` | Geographic coordinates (fallback if no `.gic`) |

**To use your own data:** set `CASE_DATA_DIR` to your extracted folder and `CASE_NAME` to match. Everything else is auto-detected.

Optionally, set `GRIDKIT_REPO` to the root of a local GridKit repository clone to augment hover labels with GridKit-assigned bus and branch IDs. JSON case files are read from `GRIDKIT_REPO/examples/PhasorDynamics/`. Set `GRIDKIT_REPO = None` to skip this augmentation.


In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set CASE_NAME, CASE_DATA_DIR, and CASE_JSON_PATH. Everything else is auto-detected.

# CASE_NAME = "Hawaii40"
# CASE_NAME = "ACTIVSg200"
CASE_NAME = "ACTIVSg2000"
# CASE_NAME = "ACTIVSg10k"

# CASE_DATA_DIR: folder containing your extracted .m, .gic, and .AUX files.
# Uncomment and set your own path:
# CASE_DATA_DIR = Path("/path/to/my/ACTIVSg200")

# My paths (Kestrel / Mac) — comment out if setting CASE_DATA_DIR above:
if onkestrel:
    CASE_DATA_DIR = (
        Path("/kfs2/projects/scidac/scidac-data") / CASE_NAME / "raw-tamu-data"
    )
else:
    CASE_DATA_DIR = (
        Path("/Users/isatkaus/projects/scidac/scidac-data")
        / CASE_NAME
        / "raw-tamu-data"
    )

# GRIDKIT_REPO: root of your local GridKit repository clone.
# Optional — set to None to skip GridKit JSON ID augmentation on hover labels.
# JSON case files live inside the GridKit repo under examples/PhasorDynamics/.
GRIDKIT_REPO = Path("/home/isatkaus/gridkit")

_case_json_paths = (
    {
        "Hawaii40": GRIDKIT_REPO / "examples/PhasorDynamics/Medium/Hawaii/hawaii.json",
        "ACTIVSg200": GRIDKIT_REPO
        / "examples/PhasorDynamics/Large/Illinois/illinois.json",
        "ACTIVSg2000": GRIDKIT_REPO / "examples/PhasorDynamics/Large/Texas/texas.json",
        "ACTIVSg10k": GRIDKIT_REPO / "examples/PhasorDynamics/Large/WECC/wecc.json",
    }
    if GRIDKIT_REPO is not None
    else {}
)
CASE_JSON_PATH = _case_json_paths.get(CASE_NAME, None)
# ────────────────────────────────────────────────────────────────────────────

case_name = CASE_NAME
case_data_dir = CASE_DATA_DIR

# File detection: set to None for auto-detect, or specify explicitly.
# Auto-detect looks for case_<name>.m first, then any single .m in case_data_dir.
m_file_name = None
# Hawaii explicit example: m_file_name = "Hawaii40_20231026.m"
gic_file_name = f"{case_name}_GIC_data.gic"
aux_file_name = None  # auto: <m_stem>.AUX/.aux or <case_name>.AUX/.aux

if m_file_name is None:
    default_m = case_data_dir / f"case_{case_name}.m"
    if default_m.exists():
        m_file_path = default_m
    else:
        m_candidates = sorted(case_data_dir.glob("*.m"))
        if len(m_candidates) == 1:
            m_file_path = m_candidates[0]
        else:
            raise FileNotFoundError(
                f"Could not uniquely determine .m file in {case_data_dir}. "
                f"Candidates: {[p.name for p in m_candidates]}"
            )
else:
    m_file_path = case_data_dir / m_file_name

gic_file_path = case_data_dir / gic_file_name

if aux_file_name is None:
    # case-insensitive: try .AUX then .aux
    def _find_aux(stem, directory):
        for ext in (".AUX", ".aux"):
            p = directory / f"{stem}{ext}"
            if p.exists():
                return p
        return directory / f"{stem}.AUX"  # canonical non-existent path for warnings

    default_aux = _find_aux(m_file_path.stem, case_data_dir)
    if default_aux.exists():
        aux_file_path = default_aux
    else:
        aux_file_path = _find_aux(case_name, case_data_dir)
else:
    aux_file_path = case_data_dir / aux_file_name

gic_exists = gic_file_path.exists()
aux_exists = aux_file_path.exists()

geo_file_path = gic_file_path if gic_exists else aux_file_path

if not m_file_path.exists():
    raise FileNotFoundError(f"MATPOWER .m file not found: {m_file_path}")
if not gic_exists:
    if aux_exists:
        print(f"INFO: GIC not found; using AUX for geo: {aux_file_path}")
    else:
        print(f"WARNING: GIC file not found at {gic_file_path}")
        print(f"WARNING: AUX file not found at {aux_file_path}")
        print("WARNING: No geo source file found (.gic or .AUX)")

print(f"case_data_dir:  {case_data_dir}")
print(f"m_file_path:    {m_file_path}")
print(f"gic_file_path:  {gic_file_path}  (exists: {gic_exists})")
print(f"aux_file_path:  {aux_file_path}  (exists: {aux_exists})")
print(f"geo_file_path:  {geo_file_path}")
print(
    f"case_json_path: {CASE_JSON_PATH}  (exists: {CASE_JSON_PATH is not None and CASE_JSON_PATH.exists()})"
)

## load MATPOWER case

Read the `.m` file into normalized `bus` / `gen` / `branch` tables via `m_viz_utils.py`.

In [ ]:
importlib.reload(m_viz_utils)
from m_viz_utils import read_matpower_case, summarize_case, validation_report_df

case_data = read_matpower_case(m_file_path)
summary = summarize_case(case_data)
validation_df = validation_report_df(case_data)

summary

## validation

These checks confirm that generator and branch bus references map back to valid `BUS_I` values.

In [ ]:
validation_df

## quick previews

Preview the normalized tables before adding geo joins or plotting logic.

In [ ]:
pd.set_option("display.max_rows", 6)
print("bus table")
case_data.bus

print("gen table")
case_data.gen

print("branch table")
case_data.branch

## optional auxiliary tables

If the MATPOWER file exposes generator fuel labels or cost tables, inspect them here.

In [ ]:
if case_data.genfuel is not None:
    print("genfuel index")
    case_data.genfuel

if case_data.gencost is not None:
    print("gencost table")
    case_data.gencost.head()

## geo merge and coordinate loading

Loads bus coordinates from `.gic` or `.AUX`, merges into the MATPOWER tables, splits colocated buses, and fans out generators that share a bus.

The geo merge result is passed to the geo plot cell below.


In [ ]:
importlib.reload(m_viz_utils)
import plotly.graph_objects as go
from m_viz_utils import attach_geo_to_case

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Circle radius (degrees lat/lon, a distance) for spreading colocated buses.
# Per-case defaults; override by setting SPLIT_RADIUS to a fixed value instead.
_split_radius_defaults = {
    "Hawaii40": 0.006,
    "ACTIVSg200": 0.006,
    "ACTIVSg2000": 0.02,
    "ACTIVSg10k": 0.04,
}
SPLIT_RADIUS = _split_radius_defaults.get(case_name, 0.006)

# Circle radius (degrees lat/lon, a distance) for spreading colocated buses.
# Per-case defaults; override by setting GEN_FANOUT_RADIUS to a fixed value instead.
_fanout_radius_defaults = {
    "Hawaii40": 0.0025,
    "ACTIVSg200": 0.004,
    "ACTIVSg2000": 0.008,
    "ACTIVSg10k": 0.015,
}
GEN_FANOUT_RADIUS = _fanout_radius_defaults.get(case_name, 0.004)

# FAN_SINGLE_GEN: offset single-generator buses by GEN_FANOUT_RADIUS (eastward)
# so the bus marker underneath remains individually hoverable.
#   False — single-gen buses stay at bus center (default; consistent across cases)
#   True  — single-gen buses also get a small eastward offset
FAN_SINGLE_GEN = False
# ────────────────────────────────────────────────────────────────────────────

# Fanout is enabled automatically when any bus has more than one generator.
n_multi_gen_buses = int((case_data.gen.groupby("GEN_BUS").size() > 1).sum())
enable_gen_fanout = n_multi_gen_buses > 0 or FAN_SINGLE_GEN
split_radius = SPLIT_RADIUS
gen_fanout_radius = GEN_FANOUT_RADIUS
fan_single_gen = FAN_SINGLE_GEN

if geo_file_path.exists():
    geo_result = attach_geo_to_case(
        case_data,
        geo_file_path,
        split_colocated=True,
        split_radius=split_radius,
        gen_fanout=enable_gen_fanout,
        gen_fanout_radius=gen_fanout_radius,
        gen_fanout_singles=fan_single_gen,
    )
    geo_case_data = geo_result.case_data

    # --- optional: attach GridKit JSON ids to hover text ---
    import gridkit_utils

    importlib.reload(gridkit_utils)
    from gridkit_utils import attach_json_ids

    case_json_path = CASE_JSON_PATH
    if case_json_path is not None and case_json_path.exists():
        attach_json_ids(geo_case_data, case_json_path)
        print(f"JSON ids attached from {case_json_path.name}")
    else:
        print("JSON ids not attached (CASE_JSON_PATH is None or file not found)")

    pd.Series(
        {
            "geo_source": str(geo_file_path),
            "split_applied": geo_result.split_applied,
            "split_radius": split_radius,
            "gen_fanout_enabled": enable_gen_fanout,
            "gen_fanout_radius": gen_fanout_radius,
            "fan_single_gen": fan_single_gen,
            "n_buses_with_multiple_gens": n_multi_gen_buses,
            "unique_bus_locations_before": geo_result.n_unique_bus_locations_before,
            "unique_bus_locations_after": geo_result.n_unique_bus_locations_after,
        }
    )
else:
    geo_result = None
    geo_case_data = None
    print(
        "Geo prep skipped: no geo source file found. "
        "Set geo_file_path to a valid .gic or .AUX file, then rerun this cell."
    )

## geo plot

Map with branches, buses (sized by PD), and generators (fuel-colored when available).

Branch **line loading** (%) = max(|S_F|, |S_T|) / RATE_A × 100, where S_F = √(PF²+QF²) and S_T = √(PT²+QT²). Requires PF, QF, PT, QT, and RATE_A columns in the `.m` branch table. Branches with RATE_A = 0 are shown in gray.

### Fault bus selection (optional)

Set `fault_bus` to highlight a bus with a red marker. Bus faults only for now (branch faults not yet supported). Set to None to skip the fault overlay entirely.

`fault_bus` accepts any of four identifiers:

| Identifier | Source | Example (Hawaii40) | Example (ACTIVSg200) |
|---|---|---|---|
| `BUS_I` (int) | `.m` file `mpc.bus` col 1 | `1` | `49` |
| `bus_name` (str) | `.m` file `mpc.bus_name` | `"ALOHA138"` | `"RANTOUL 2 1"` |
| JSON bus number (int) | `"number"` field in JSON `Bus` device | `1` (same as `BUS_I`) | `49` (same as `BUS_I`) |
| JSON bus name (str) | `"name"` field in JSON `Bus` device | `"ALOHA138"` (same as `bus_name`) | `"RANTOUL 2 1"` (same) |

For both Hawaii40 and ACTIVSg200, the JSON `"number"` and `"name"` fields match the `.m` file's `BUS_I` and `bus_name` exactly, so all four forms are equivalent.


In [ ]:
importlib.reload(m_viz_utils)
from m_viz_utils import plot_grid, lookup_fault_bus

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# SHOW_LOADING = False  # False: uniform gray branches
SHOW_LOADING = True  # True: viridis-colored branches by loading_pct

# SAVE_HTML = False
SAVE_HTML = True

# SAVE_FIGS_DIR: directory where HTML figures are saved.
# Uncomment and set your own path:
# SAVE_FIGS_DIR = Path("/path/to/my/figs")

# My paths (Kestrel / Mac):
if onkestrel:
    SAVE_FIGS_DIR = Path("/home/isatkaus/gridkit/uq-usecase/figs")
else:
    SAVE_FIGS_DIR = Path("/Users/isatkaus/projects/scidac/scidac-data/figs")

# FIG_WIDTH / FIG_HEIGHT: figure size in pixels. None = plot_grid defaults (width=780, height=650).
FIG_WIDTH = 920
FIG_HEIGHT = None

# MARKER_SCALE: global multiplier for all circle marker sizes (buses and generators).
#   1.0  — default sizes
#   0.5  — half size (useful for large cases with many overlapping markers)
#   2.0  — double size
MARKER_SCALE = 1.0

# FAULT_BUS: bus to highlight with a red fault marker. Set to None to skip.
# Bus faults only for now (branch faults not yet supported).
# Accepts any of four identifiers (all equivalent for Hawaii40 and ACTIVSg200):
#   BUS_I (int)     from .m file mpc.bus col 1   e.g. 1          or 49
#   bus_name (str)  from .m file mpc.bus_name     e.g. "ALOHA138" or "RANTOUL 2 1"
#   JSON "number"   Bus device "number" field      e.g. 1          or 49   (= BUS_I)
#   JSON "name"     Bus device "name" field        e.g. "ALOHA138" or "RANTOUL 2 1" (= bus_name)
_fault_defaults = {
    "Hawaii40": "ALOHA138",
    "ACTIVSg200": 147,  # parametric — all 200 buses are wired with BusFault devices
    "ACTIVSg2000": 6349,  ### 6349, 7428, 7422
    "ACTIVSg10k": 100,
}
FAULT_BUS = _fault_defaults.get(case_name, None)

# INCLUDE_PLOTLYJS: controls how plotly.js is included in the saved HTML.
#   True    — bundles plotly.js (~3 MB); required for MkDocs iframes and offline viewing (default)
#   "cdn"   — loads plotly.js from CDN at open time; ~50 KB file, but requires internet access
INCLUDE_PLOTLYJS = True
# INCLUDE_PLOTLYJS = "cdn"
# ────────────────────────────────────────────────────────────────────────────

show_loading = SHOW_LOADING
save_html = SAVE_HTML
save_figs_dir = SAVE_FIGS_DIR
fig_width = FIG_WIDTH
fig_height = FIG_HEIGHT
marker_scale = MARKER_SCALE
fault_bus = FAULT_BUS

# Show gen connectors whenever gen fanout was enabled in the geo merge cell.
show_gen_connectors = enable_gen_fanout

if geo_case_data is not None:
    fig_geo = plot_grid(
        geo_case_data,
        zoom=4,
        show_loading=show_loading,
        show_gen_connectors=show_gen_connectors,
        marker_scale=marker_scale,
    )
    if fig_width is not None or fig_height is not None:
        _ = fig_geo.update_layout(
            width=fig_width if fig_width is not None else fig_geo.layout.width,
            height=fig_height if fig_height is not None else fig_geo.layout.height,
        )

    # --- fault bus overlay ---
    if fault_bus is not None:
        bus_row = lookup_fault_bus(geo_case_data.bus, fault_bus)
        if bus_row.empty:
            bus_df = geo_case_data.bus
            print(
                f"WARNING: fault_bus={fault_bus!r} not found.\n"
                f"Available bus_name values: {bus_df['bus_name'].tolist() if 'bus_name' in bus_df.columns else 'N/A'}\n"
                f"BUS_I range: {bus_df['BUS_I'].min()} – {bus_df['BUS_I'].max()}"
            )
        else:
            fault_lat = float(bus_row["lat"].iloc[0])
            fault_lon = float(bus_row["lon"].iloc[0])
            fault_bus_i = int(bus_row["BUS_I"].iloc[0])
            fault_label = (
                bus_row["bus_name"].iloc[0]
                if "bus_name" in bus_row.columns
                else str(fault_bus_i)
            )
            fault_group = f"fault_{fault_bus_i}"

            _ = fig_geo.add_trace(
                go.Scattermap(
                    mode="markers",
                    lat=[fault_lat],
                    lon=[fault_lon],
                    hoverinfo="skip",
                    marker=dict(size=36, color="black", opacity=1.0),
                    legendgroup=fault_group,
                    showlegend=False,
                )
            )
            _ = fig_geo.add_trace(
                go.Scattermap(
                    name=f"⚡ fault: {fault_label}",
                    mode="markers",
                    lat=[fault_lat],
                    lon=[fault_lon],
                    hovertext=[
                        f"<b>⚡ FAULT</b><br>BUS_I: {fault_bus_i}<br>Name: {fault_label}"
                    ],
                    hoverinfo="text",
                    marker=dict(size=26, color="red", opacity=1.0),
                    legendgroup=fault_group,
                    showlegend=True,
                )
            )

    if save_html:
        save_figs_dir.mkdir(parents=True, exist_ok=True)
        suffix = "_loading" if show_loading else ""
        fault_suffix = f"_fault{fault_bus}" if fault_bus is not None else ""
        html_stem = f"{case_name}_geo{suffix}{fault_suffix}"
        html_path = save_figs_dir / f"{html_stem}.html"
        # height=None + viewport CSS fills browser window in new tab
        fig_html = go.Figure(fig_geo)
        _ = fig_html.update_layout(height=None, width=None, autosize=True)
        _viewport_css = (
            "var s=document.createElement('style');"
            "s.textContent='html,body{height:100vh;margin:0;padding:0;overflow:hidden;}';"
            "document.head.appendChild(s);"
        )
        fig_html.write_html(
            str(html_path),
            full_html=True,
            include_plotlyjs=INCLUDE_PLOTLYJS,
            post_script=_viewport_css,
        )
        print(f"figure saved to {html_path}")

    fig_geo
else:
    print(
        "Geo plot skipped because geo_case_data is not available. "
        "Provide a valid geo_file_path and rerun geo prep cell first."
    )

In [ ]:
#### check file sizes for notebook and saved HTML
def _fmt_mb(path):
    mb = Path(path).stat().st_size / 1024**2
    return f"{mb:.2f} MB  ({path})"


nb_path = Path.cwd() / "m_viz.ipynb"
print(_fmt_mb(nb_path))

if save_html and html_path.exists():
    print(_fmt_mb(html_path))

In [ ]:
# importlib.reload(m_viz_utils)
# from m_viz_utils import plot_grid, lookup_fault_bus

# # --- plot options ---
# show_loading = (
#     True  # True: viridis-colored branches by loading_pct (requires PF/QF/PT/QT/RATE_A)
# )
# # show_loading = False  # False: uniform gray branches

# # save_html = False  # True: write figure to save_figs_dir as .html
# save_html = True

# save_figs_dir = scidac_data_dir / "figs"  # adjust as needed
# # save_figs_dir = Path("/home/isatkaus/projects/scidac/isatkaus/scidac-notebooks/figs")

# show_gen_connectors = "hawaii" in str(case_name).lower()

# # --- fault bus (optional) ---
# # Identify a faulted bus by BUS_I (int) or bus_name (str, from mpc.bus_name).
# # Set to None to skip.
# fault_bus = None
# # fault_bus = 1           # ACTIVSg200: by BUS_I integer
# fault_bus = "ALOHA138"  # Hawaii40: by bus_name string

# if geo_case_data is not None:
#     fig_geo = plot_grid(
#         geo_case_data,
#         zoom=7,
#         show_loading=show_loading,
#         show_gen_connectors=show_gen_connectors,
#     )

#     # --- fault bus overlay ---
#     if fault_bus is not None:
#         bus_row = lookup_fault_bus(geo_case_data.bus, fault_bus)
#         if bus_row.empty:
#             bus_df = geo_case_data.bus
#             print(
#                 f"WARNING: fault_bus={fault_bus!r} not found.\n"
#                 f"Available bus_name values: {bus_df['bus_name'].tolist() if 'bus_name' in bus_df.columns else 'N/A'}\n"
#                 f"BUS_I range: {bus_df['BUS_I'].min()} – {bus_df['BUS_I'].max()}"
#             )
#         else:
#             fault_lat = float(bus_row["lat"].iloc[0])
#             fault_lon = float(bus_row["lon"].iloc[0])
#             fault_bus_i = int(bus_row["BUS_I"].iloc[0])
#             fault_label = (
#                 bus_row["bus_name"].iloc[0]
#                 if "bus_name" in bus_row.columns
#                 else str(fault_bus_i)
#             )
#             fault_group = f"fault_{fault_bus_i}"

#             _ = fig_geo.add_trace(
#                 go.Scattermap(
#                     mode="markers",
#                     lat=[fault_lat],
#                     lon=[fault_lon],
#                     hoverinfo="skip",
#                     marker=dict(size=36, color="black", opacity=1.0),
#                     legendgroup=fault_group,
#                     showlegend=False,
#                 )
#             )
#             _ = fig_geo.add_trace(
#                 go.Scattermap(
#                     name=f"⚡ fault: {fault_label}",
#                     mode="markers",
#                     lat=[fault_lat],
#                     lon=[fault_lon],
#                     hovertext=[
#                         f"<b>⚡ FAULT</b><br>BUS_I: {fault_bus_i}<br>Name: {fault_label}"
#                     ],
#                     hoverinfo="text",
#                     marker=dict(size=26, color="red", opacity=1.0),
#                     legendgroup=fault_group,
#                     showlegend=True,
#                 )
#             )

#     if save_html:
#         save_figs_dir.mkdir(parents=True, exist_ok=True)
#         suffix = "_loading" if show_loading else ""
#         fault_suffix = f"_fault{fault_bus}" if fault_bus is not None else ""
#         html_stem = f"{case_name}_geo{suffix}{fault_suffix}"
#         html_path = save_figs_dir / f"{html_stem}.html"
#         # height=None → 100% of container; fills browser window in new tab,
#         # fits the iframe height in the MkDocs page without internal scrollbars
#         fig_html = go.Figure(fig_geo)
#         _ = fig_html.update_layout(height=None)
#         fig_html.write_html(str(html_path), full_html=True, include_plotlyjs=True)
#         print(f"figure saved to {html_path}")

#     fig_geo
# else:
#     print(
#         "Geo plot skipped because geo_case_data is not available. "
#         "Provide a valid geo_file_path and rerun geo prep cell first."
#     )

In [ ]:
show_gen_connectors = "hawaii" in str(case_name).lower()

# --- figure size (pixels; None = use plot_grid defaults: width=780, height=650) ---
fig_width = None
fig_height = None